In [1]:
import os
import platform
import sys
import time
import json

from dataclasses import asdict

import numpy as np
import h5py
import pint
import tables

from pint import Quantity

In [2]:
match platform.system():
    case "Linux":
        sys.path.insert(1, os.path.abspath(".."))
        import lysis
        from lysis.util import Q_
        lysis_root = os.path.join("/", "home", "bpaynter", "git", "UCO-OpResearch", "lysis")
    case "Windows":
        import src.python.lysis as lysis

In [3]:
#Need to make considerations for loading the data from an existing HDF5 instead of just loading a new one every time.
#Need to start moving this notebook to lysis\src\python\lysis\util\ and replace datastore.py with this code becoming methods.
#Need to create some sort of class
run_code="2024-09-02-1411"
r = lysis.util.Run(os.path.join(lysis_root, "data"), run_code=run_code)
r.read_file()
print(r)

{'fibrinogen_length': <Quantity(0.045, 'micron')>, 'fibrinogen_radius': <Quantity(0.0012, 'micron')>, 'fiber_radius': <Quantity(0.0615, 'micron')>, 'diss_const_tPA_wPLG': <Quantity(0.02, 'micromolar')>, 'diss_const_tPA_woPLG': <Quantity(0.36, 'micromolar')>, 'diss_const_PLG_intact': <Quantity(38, 'micromolar')>, 'diss_const_PLG_nicked': <Quantity(2.2, 'micromolar')>, 'bind_rate_tPA': <Quantity(0.1, '1 / micromolar / second')>, 'bind_rate_PLG': <Quantity(0.1, '1 / micromolar / second')>, 'conc_free_PLG': <Quantity(2, 'micromolar')>, 'deg_rate_fibrin': <Quantity(5.0, '1 / second')>, 'unbind_rate_PLi': <Quantity(57.6, '1 / second')>, 'activation_rate_PLG': <Quantity(0.1, '1 / second')>, 'exposure_rate_binding_site': <Quantity(5.0, '1 / second')>, 'nodes_in_micro_row': 13, 'snap_proportion': 0.6666666666666666, 'micro_simulations': 50000, 'micro_seed': 2133256963, 'micro_version': 'micro_rates', 'log_lvl': 30}
{'pore_size': <Quantity(0.000534, 'centimeter')>, 'diffusion_coeff': <Quantity(5

## Test for read and export

In [4]:
# data = lysis.util.DataStore(run_code, os.path.join(lysis_root, "data"), 'r')
# data.export_fortran_micro_data(os.path.join(lysis_root, "data", run_code), "__hdf5")

Creates a variable f that references the file we want to access.
Need to change variables according to the name of you .h5 file

In [5]:
h5py.get_config().track_order = True
h5file = h5py.File(os.path.join(lysis_root, "data", f'{run_code}.h5'), 'w')

## Building Import

In [6]:
tpa_molecule_status = h5py.enum_dtype({i.name: i.value for i in lysis.util.CONST.MOL_STATUS}, basetype="u1")

In [ ]:
fortran_tpa_bind_event_type = np.dtype([
    ("Simulation Time Elapsed", np.float64), 
    ("tPA Molecule Index", np.int32), 
    ("Molecule New Status", np.int32),
    ("Grid Location Index", np.int32)
])

hdf5_tpa_bind_event_type = np.dtype([
    ("Simulation Time Elapsed", np.float64), 
    ("tPA Molecule Index", np.int64), 
    ("Molecule New Status", tpa_molecule_status),
    ("Grid Location Row", np.uint32),
    ("Grid Location Rank", np.uint32),
])

fortran_fiber_degrade_event_type = np.dtype([
    ("Simulation Time Elapsed", np.float64),
    ("Grid Location Index", np.int32),
    ("Fiber New Degrade Time", np.float64),
])

hdf5_fiber_degrade_event_type = np.dtype([
    ("Simulation Time Elapsed", np.float64),
    ("Grid Location Row", np.uint32),
    ("Grid Location Rank", np.uint32),
    ("Fiber New Degrade Time", np.float64),
])

Creates folder and data set structure

In [8]:
# runs = h5file.create_group("1-PKd") 
micro_data = h5file.create_group("micro_data") 
macro_data = h5file.create_group("macro_data")



#Micro data set initializations
pli_first_time = micro_data.create_dataset("pli_first_time", (r.micro_params.simulations, ), dtype= np.float64, compression="gzip")
tpa_final_num = micro_data.create_dataset("tpa_final_num", (r.micro_params.simulations, ), dtype= np.uint8, compression="gzip")
fiber_degraded = micro_data.create_dataset("fiber_degraded", (r.micro_params.simulations, ), dtype= np.bool_, compression="gzip")
sim_final_time = micro_data.create_dataset("sim_final_time", (r.micro_params.simulations, ), dtype= np.float64, compression="gzip")
pli_generated_num = micro_data.create_dataset("pli_generated_num", (r.micro_params.simulations, ), dtype= np.uint16, compression="gzip")
tpa_leaving_time = micro_data.create_dataset("tpa_leaving_time", (r.micro_params.simulations, ), dtype= np.float64, compression="gzip")
tpa_unbound_by_pli = micro_data.create_dataset("tpa_unbound_by_pli", (r.micro_params.simulations, ), dtype= np.bool_, compression="gzip")
tpa_unbound_kinetic = micro_data.create_dataset("tpa_unbound_kinetic", (r.micro_params.simulations, ), dtype = np.bool_, compression="gzip")

#Macro data set initializations


for i in range(0 , r.macro_params.simulations):
    simulation = macro_data.create_group(f"sim_{i:02}")
    fiber_degrade_time = simulation.create_dataset("fiber_degrade_time", (1,) , dtype=hdf5_fiber_degrade_event_type, maxshape = (None,), compression="gzip", chunks=(10_000,))
    tpa_bind_events = simulation.create_dataset("tpa_bind_events", (1,) , dtype=hdf5_tpa_bind_event_type , maxshape = (None,), compression="gzip", chunks=(10_000,)) #snapshot time = 
    snapshot_time = simulation.create_dataset("snapshot_time", (1,) , dtype=np.float64 , maxshape = (None,), compression="gzip", chunks=(10_000,))
    #tpa_location_snapshot = simulation.create_dataset("tpa_location_snapshot", (r.macro_params.total_molecules , 1) , dtype=np.int32 , maxshape = (r.macro_params.total_molecules, None)) #I uncapped the max number of rows so the data can fit
    tpa_location_snapshot = simulation.create_dataset("tpa_location_snapshot", (r.macro_params.total_molecules, 2, 1) , dtype=np.int32, maxshape = (r.macro_params.total_molecules, 2, None), compression="gzip", chunks=(r.macro_params.total_molecules, 2, 5))
    tpa_transit_time = simulation.create_dataset("tpa_transit_time", (r.macro_params.total_molecules,) , dtype=np.float64, compression="gzip")
    


Reads in Micro scale data into file system

In [9]:
micro_file_code = "_PLG2_tPA01_TB-xiii"

pli_first_time[:] = np.fromfile(
    os.path.join(r.os_path, f"firstPLi{micro_file_code}.dat"),
)
pli_first_time.attrs["units"] = "seconds"

tpa_final_num[:] = np.fromfile(os.path.join(r.os_path, f"lasttPA{micro_file_code}.dat"), dtype=np.int32)
tpa_final_num.attrs["units"] = "none"

fiber_degraded[:] = np.fromfile(
    os.path.join(r.os_path, f"lyscomplete{micro_file_code}.dat"), 
    dtype=np.int32
).astype(bool)
fiber_degraded.attrs["units"] = "none"

sim_final_time[:] = np.fromfile(os.path.join(r.os_path, f"lysis{micro_file_code}.dat"))
sim_final_time.attrs["units"] = "seconds"

pli_generated_num[:] = np.fromfile(os.path.join(r.os_path, f"PLi{micro_file_code}.dat"), dtype=np.int32).astype(np.uint16)
pli_generated_num.attrs["units"] = "none"

tpa_leaving_time[:] = np.fromfile(os.path.join(r.os_path, f"tPA_time{micro_file_code}.dat"))
tpa_leaving_time.attrs["units"] = "seconds"

tpa_unbound_by_pli[:] = np.fromfile(
    os.path.join(r.os_path, f"tPAPLiunbd{micro_file_code}.dat"), 
    dtype=np.int32
).astype(bool)
tpa_unbound_by_pli.attrs["units"] = "none"

tpa_unbound_kinetic[:] = np.fromfile(
    os.path.join(r.os_path, f"tPAunbind{micro_file_code}.dat"), 
    dtype=np.int32
).astype(bool)
tpa_unbound_kinetic.attrs["units"] = "none"

Reads in Macro scale Fiber Degrade Time Data

In [10]:
macro_file_code = f"_TB-xiii__21_105"
for i in range (0 , 10):
    file_reference = h5file[f"macro_data/sim_{i:02}/fiber_degrade_time"]
    raw = np.loadtxt(os.path.join(r.os_path, f"{i:02}", f"f_deg_list{macro_file_code}_{i:02}.dat") , delimiter=",", dtype=fortran_fiber_degrade_event_type)
    file_reference.resize(raw.shape)
    data = np.empty(raw.shape, dtype=hdf5_fiber_degrade_event_type)
    data[["Simulation Time Elapsed", "Fiber New Degrade Time"]] = raw[["Simulation Time Elapsed", "Fiber New Degrade Time"]]
    data[["Grid Location Row", "Grid Location Rank"]] = [lysis.util.from_fortran_edge_index(idx-1, r.macro_params.rows, r.macro_params.cols) for idx in raw["Grid Location Index"]]
    #Adding Attributes to datasets
    file_reference[:] = data
    file_reference.attrs["units"] = ["seconds" , "none", "none" , "seconds"]

Reads in Macro scale TPA Bind Events Data | ~~Needs to be reworked for efficiency 51 seconds (5.5 min on Buddy)~~ Fixing chunk size took care of this

I believe what is taking so long is the resizing of the HDF dataset that is going on. Perhaps by specifying size on instatiation, we can avoid this.

It was the writing to the HDF itself. When resizing is enabled, chunking is enabled. With the initial size set to '1', the chunk size was also set to 1. This made writing horribly inefficient as it created a new chunk for each row.

In [11]:
for i in range (0 , 10):
    file_reference = h5file[f"macro_data/sim_{i:02}/tpa_bind_events"]
    raw = np.loadtxt(os.path.join(r.os_path, f"{i:02}", f"m_bind_t{macro_file_code}_{i:02}.dat") , delimiter=",", dtype=fortran_tpa_bind_event_type)
    file_reference.resize(raw.shape)
    data = np.empty(raw.shape, dtype=hdf5_tpa_bind_event_type)
    data[["Simulation Time Elapsed", "Molecule New Status"]] = raw[["Simulation Time Elapsed", "Molecule New Status"]]
    # The data in column 1 and 3 is 1 indexed, so we need to convert it to 0 indexed
    data['tPA Molecule Index'] = raw['tPA Molecule Index'] - 1
    data[["Grid Location Row", "Grid Location Rank"]] = [lysis.util.from_fortran_edge_index(idx-1, r.macro_params.rows, r.macro_params.cols) for idx in raw["Grid Location Index"]]
    file_reference[:] = data
    file_reference.attrs["units"] = ["seconds" , "none" , "none", "none", "none"]
    

Reads in Macro scale snapshot time data

In [12]:
for i in range (0 , 10):
    file_reference = h5file[f"macro_data/sim_{i:02}/tpa_location_snapshot"]
    raw = np.fromfile(os.path.join(r.os_path, f"{i:02}", f"m_loc{macro_file_code}_{i:02}.dat") , dtype = np.int32).reshape(r.macro_params.total_molecules, -1)
    data = np.empty((raw.shape[0], 2, raw.shape[1]), dtype=np.int32)
    for idx, x in np.ndenumerate(raw):
        data[idx[0], :, idx[1]] = lysis.util.from_fortran_edge_index(x-1, r.macro_params.rows, r.macro_params.cols) 
    file_reference.resize(data.shape)
    file_reference[:] = data
    file_reference.attrs["units"] = 'none'

Reads in Macro scale TPA Transit Time Data

In [13]:
for i in range (0 , 10):
    file_reference = h5file[f"macro_data/sim_{i:02}/tpa_transit_time"]
    data = np.fromfile(os.path.join(r.os_path, f"{i:02}", f"mfpt{macro_file_code}_{i:02}.dat") , dtype = np.float64) #.reshape(-1,1)
    file_reference[:] = data
    file_reference.attrs["units"] = "seconds"

Reads in Macro scale Snapshot Time Data

In [14]:
for i in range (0 , 10):
    file_reference = h5file[f"macro_data/sim_{i:02}/snapshot_time"]
    data = np.fromfile(os.path.join(r.os_path, f"{i:02}", f"tsave{macro_file_code}_{i:02}.dat") , dtype = np.float64) #.reshape(-1,1)
    file_reference.resize(data.shape)
    file_reference[:] = data
    file_reference.attrs["units"] = "seconds"

Adding units to group attributes

In [15]:
params = r.to_dict()

In [16]:
micro_group = h5file["micro_data"]
units = lysis.util.MicroParameters.units()
for k, v in params["micro_params"].items():
    if isinstance(v, Quantity):
        micro_group.attrs[k] = str(v.to(units[k]))
    else:
        micro_group.attrs[k] = v



In [17]:
macro_group = h5file["macro_data"]
units = lysis.util.MacroParameters.units()
for k, v in params["macro_params"].items():
    if isinstance(v, Quantity):
        macro_group.attrs[k] = str(v.to(units[k]))
    else:
        macro_group.attrs[k] = v

Create group for log files

In [18]:
log_group = h5file.create_group("log_files")
with open(os.path.join(r.os_path, f"micro{micro_file_code}.txt"), 'r') as file:
    micro_log = file.readlines()
micro_log_dataset = log_group.create_dataset("micro_log", (1,) , dtype=h5py.string_dtype() , maxshape = (None,), compression="gzip", chunks=(10_000,))
micro_log_dataset.resize((len(micro_log),))
micro_log_dataset[:] = micro_log

In [19]:
for i in range(r.macro_params.simulations):
    macro_log_dataset = log_group.create_dataset(f"macro_log__sim_{i:02}", (1,) , dtype=h5py.string_dtype() , maxshape = (None,), compression="gzip", chunks=(10_000,))
    with open(os.path.join(r.os_path, f"{i:02}", f"macro{macro_file_code}_{i:02}.txt"), 'r') as file:
        macro_log = file.readlines()
    macro_log_dataset.resize((len(macro_log),))
    macro_log_dataset[:] = macro_log

In [20]:
h5file.close()

## Test code for "Micro to Macro" conversions.

In [ ]:
h5file = h5py.File(os.path.join(lysis_root, "data", f'{run_code}.h5'), 'r')

In [ ]:
micro_data = h5file["micro_data"]
set_size = micro_data["pli_first_time"].size // 100
tPAleave = np.append(np.arange(0, 1, 0.01), [1.0])
np.savetxt(os.path.join(r.os_path, "tPAleave_numpy.dat"), tPAleave)

In [ ]:
# tPA_leave_time = np.fromfile(os.path.join(e.os_path, f"tPA_time_{file_code}.dat"))
indices = micro_data["tpa_leaving_time"][:].argsort()
tsectPA = np.append(
    [0], micro_data["tpa_leaving_time"][:][indices[set_size - 1 :: set_size]]
)
np.savetxt(os.path.join(r.os_path, "tsectPA_numpy.dat"), tsectPA)

In [ ]:
lysis_complete = micro_data["fiber_degraded"][:]
lysis_time = micro_data["sim_final_time"][:]
lysis_time[~lysis_complete] = 6000
lysismat = np.stack(
    [
        np.sort(lysis_time[indices[i * set_size : (i + 1) * set_size]])
        for i in range(100)
    ]
).T
np.savetxt(os.path.join(r.os_path, "lysismat_numpy.dat"), lysismat)

In [ ]:
lenlysisvect = lysismat.argmax(axis=0)+1
np.savetxt(os.path.join(r.os_path, "lenlysisvect_numpy.dat"), lenlysisvect)

In [ ]:
h5file.close()

## Pint Testing

Post Processing

In [ ]:
import pint
u = pint.UnitRegistry()
Q = u.Quantity

In [13]:
h5file = h5py.File(os.path.join(lysis_root, "data", f'{run_code}.h5'), 'a')

In [ ]:
for k, v in h5file.items():
    print(k)

In [14]:
dataset = h5file["log_files/micro_log"]
numpy_array = dataset[:]
dataset[0] = "This is a test!"

In [ ]:
H5_dataset = h5file["macro_data/sim_00/fiber_degrade_time"]
unit_array = H5_dataset.attrs["units"]
dataset = np.array(h5file["macro_data/sim_00/fiber_degrade_time"])
event_time = dataset[:,0]
legs2 = [400.0, 300.0] * u.centimeter
legs2 = event_time * u(unit_array[0])
print(legs2.to('min'))
#print(legs2)
unit_array

In [ ]:
data = h5file["micro_data/tpa_final_num"]
data.attrs["units"]

In [ ]:
r.macro_params

## DataStore Testing

In [4]:
data = lysis.util.read_data_collection(r.os_path, [lysis.util.dataspec["v1.99.0"].microscale_out, lysis.util.dataspec["v1.99.0"].macroscale_out], file_codes=["_PLG2_tPA01_TB-xiii", "_TB-xiii__21_105"])
data.keys()

/home/bpaynter/git/UCO-OpResearch/lysis/src/python/lysis/util/fileops.py:40: UserWarning: Input line 2 contained no data and will not be counted towards `max_rows=50000`.  This differs from the behaviour in NumPy <=1.22 which counted lines rather than rows.  If desired, the previous behaviour can be achieved by using `itertools.islice`.
Please see the 1.23 release notes for an example on how to do this.  If you wish to ignore this warning, use `warnings.filterwarnings`.  This warning is expected to be removed in the future and is given only once per `loadtxt` call.
  return np.loadtxt(


dict_keys(['params', 'micro', 'micro_rates', 'firstPLi', 'lasttPA', 'lyscomplete', 'lysis', 'PLi', 'tPA_time', 'tPAPLiunbd', 'tPAunbind', 'macro', 'Nsave', 'tsave', 'f_deg_list', 'm_bind_t', 'm_loc', 'm_bound', 'mfpt'])

In [5]:
data["params"]

{'run_code': '2024-09-02-1411',
 'data_filenames': None,
 'micro_params': {'fibrinogen_length': '0.045 micron',
  'fibrinogen_radius': '0.0012 micron',
  'fiber_radius': '0.0615 micron',
  'protofibril_radius': '0.0024 micron',
  'diss_const_tPA_wPLG': '0.02 micromolar',
  'diss_const_tPA_woPLG': '0.36 micromolar',
  'diss_const_PLG_intact': '38 micromolar',
  'diss_const_PLG_nicked': '2.2 micromolar',
  'bind_rate_tPA': '0.1 / micromolar / second',
  'bind_rate_PLG': '0.1 / micromolar / second',
  'conc_free_PLG': '2 micromolar',
  'deg_rate_fibrin': '5.0 / second',
  'unbind_rate_PLG_intact': '3.8000000000000003 / second',
  'unbind_rate_PLG_nicked': '0.22000000000000003 / second',
  'unbind_rate_PLi': '57.6 / second',
  'unbind_rate_tPA_wPLG': '0.002 / second',
  'unbind_rate_tPA_woPLG': '0.036 / second',
  'activation_rate_PLG': '0.1 / second',
  'exposure_rate_binding_site': '5.0 / second',
  'protein_per_fiber': '25.737061273051754 percent',
  'fibrin_conc_per_fiber': '1049.67095

In [9]:
a = {"b": 1, "c": 3}
def func(b=4, c=10):
    print(b, c)

func(**None)

TypeError: __main__.func() argument after ** must be a mapping, not NoneType